# ⚙️ CI/CD for Computational Toxicology
## Continuous Integration & Continuous Deployment — A Practical Tutorial
### Python Ecosystem Tutorial Series — Bonus Module

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

| | |
|---|---|
| **Concept** | CI/CD (Continuous Integration / Continuous Deployment) |
| **Domain** | Scientific Software Engineering + Computational Toxicology |
| **Tools** | GitHub Actions, pytest, Docker, MLflow, FastAPI |
| **Example pipeline** | QSAR toxicity model: code → test → build → deploy → monitor |

## What you will learn

1. What CI/CD is and why computational scientists need it
2. How to write testable scientific code (pytest for cheminformatics)
3. How to build a GitHub Actions CI pipeline that runs on every commit
4. How to containerise a toxicity prediction model with Docker
5. How to version models with MLflow and deploy via FastAPI
6. A complete end-to-end worked example: from SMILES → prediction → API

```bash
pip install pytest pytest-cov rdkit-pypi scikit-learn pandas numpy
pip install fastapi uvicorn mlflow docker requests
```

---
## 1. What is CI/CD? (And why should a computational scientist care?)

### The Problem CI/CD Solves

Imagine you're developing a QSAR toxicity model. Your workflow today might look like:

```
Write code → Run it manually → Email results → Hope it still works next month
```

This breaks because:
- A collaborator changes a SMILES parser and silently breaks your predictions
- You retrain the model on new data — did the AUC go up or down vs last week?
- You deploy a fix to the API server but forgot to update the Docker image
- Your paper's results can't be reproduced because the software version isn't tracked

**CI/CD solves all of this.** It automates the boring, critical steps so they happen
on every single code change — not just when you remember.

---

### The Two Halves

```
  Developer           GitHub / GitLab        Your Server / Cloud
  pushes code    →    CI runs tests     →    CD deploys if tests pass
  
  "Continuous         "Continuous            "Continuous
   Integration"        Integration"           Deployment"
```

**Continuous Integration (CI):**  
Every time code is pushed, automatically: lint → unit test → integration test → coverage report.  
If anything fails, the team is notified immediately and the bad code is never merged.

**Continuous Deployment (CD):**  
If CI passes, automatically: build Docker image → push to registry → deploy to server → health check.  
Zero manual deployment steps.

---

### CI/CD vs Traditional Scientific Workflow

| Step | Without CI/CD | With CI/CD |
|------|---------------|------------|
| Test new code | Manually, if at all | Automatic on every push |
| Catch regressions | When someone notices | Immediately, before merge |
| Deploy model update | SSH in, run commands, pray | Automatic, versioned, reversible |
| Reproduce results | "It worked on my machine" | Docker image, pinned versions |
| Monitor model drift | Never | Automated alerts |


---
## 2. Our Example: A QSAR Acute Toxicity Prediction Pipeline

We'll build a complete, CI/CD-ready toxicity prediction system with these components:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    Toxicity Prediction Pipeline                              │
│                                                                             │
│  Data Layer           Model Layer          API Layer         Monitoring     │
│  ──────────           ───────────          ─────────         ──────────     │
│  Raw SMILES      →    RDKit features  →   FastAPI       →   MLflow         │
│  ToxCast data         RF/GBM model        /predict         model registry  │
│  (Tox21 subset)       AUC ≥ 0.80         /health          drift alerts     │
│                       MLflow tracking     /docs                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│                    CI/CD Pipeline (GitHub Actions)                          │
│                                                                             │
│  git push                                                                   │
│     ↓                                                                       │
│  [Trigger]                                                                  │
│     ↓                                                                       │
│  Stage 1: Lint          Stage 2: Unit Tests      Stage 3: Integration      │
│  ─────────────          ────────────────────      ───────────────────────  │
│  flake8 src/            pytest tests/unit/        pytest tests/integration/ │
│  black --check          pytest-cov ≥ 80%          Test full API lifecycle   │
│     ↓                        ↓                           ↓                 │
│  Stage 4: Build              Stage 5: Deploy       Stage 6: Health Check   │
│  ──────────────              ───────────────       ──────────────────────  │
│  docker build                docker push           curl /health → 200 OK   │
│  docker scan (CVE)           k8s rollout           AUC ≥ 0.80 threshold    │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Repository Structure
```
toxicity-qsar/
├── .github/
│   └── workflows/
│       ├── ci.yml          # runs on every push/PR
│       └── cd.yml          # runs on merge to main
├── src/
│   ├── featurize.py        # SMILES → RDKit descriptors
│   ├── model.py            # training + evaluation
│   └── api.py              # FastAPI prediction service
├── tests/
│   ├── unit/
│   │   ├── test_featurize.py
│   │   └── test_model.py
│   └── integration/
│       └── test_api.py
├── Dockerfile
├── requirements.txt
└── mlflow_config.yml
```


---
## 3. Writing Testable Scientific Code

The foundation of CI/CD is **testable code**. This means structuring functions
so each one does one thing and can be verified independently.

### Before: Untestable monolith (common in science)
```python
# BAD: one giant script — can't test individual steps
df = pd.read_csv("chemicals.csv")
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, 1024) for m in mols]
X = np.array(fps)
rf = RandomForestClassifier(n_estimators=100).fit(X, df["toxic"])
preds = rf.predict(X)
print(f"Accuracy: {(preds == df['toxic']).mean():.3f}")
```

### After: Testable functions
```python
# GOOD: each function does one thing and can be tested in isolation
def smiles_to_mol(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles!r}")
    return mol

def mol_to_fingerprint(mol, radius: int = 2, n_bits: int = 1024):
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, n_bits)

def featurize_smiles(smiles: str) -> np.ndarray:
    mol = smiles_to_mol(smiles)
    fp = mol_to_fingerprint(mol)
    return np.array(fp)
```

Now you can write a test for each function:
```python
def test_smiles_to_mol_valid():
    mol = smiles_to_mol("CC(=O)Oc1ccccc1C(=O)O")  # aspirin
    assert mol is not None

def test_smiles_to_mol_invalid_raises():
    with pytest.raises(ValueError, match="Invalid SMILES"):
        smiles_to_mol("NOT_A_SMILES!!!")

def test_featurize_returns_correct_shape():
    fp = featurize_smiles("c1ccccc1")  # benzene
    assert fp.shape == (1024,)
    assert fp.dtype == np.float64
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── Source Module 1: featurize.py ─────────────────────────────────────────────
# This is what lives in src/featurize.py in the repository

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
    RDKIT_OK = True
except ImportError:
    RDKIT_OK = False
    print("RDKit not available — using fingerprint simulation")

import hashlib

def smiles_to_mol(smiles: str):
    """Convert SMILES string to RDKit molecule object.
    
    Args:
        smiles: SMILES representation of a molecule
        
    Returns:
        RDKit Mol object
        
    Raises:
        ValueError: if SMILES is invalid
    """
    if not RDKIT_OK:
        return smiles  # passthrough when RDKit not available
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles!r}")
    return mol


def mol_to_morgan_fp(mol, radius: int = 2, n_bits: int = 1024) -> np.ndarray:
    """Compute Morgan (ECFP) fingerprint as bit vector.
    
    Args:
        mol: RDKit Mol object
        radius: Morgan radius (2 = ECFP4)
        n_bits: fingerprint length
        
    Returns:
        numpy array of shape (n_bits,)
    """
    if not RDKIT_OK:
        # Deterministic pseudo-fingerprint from SMILES hash (for testing without RDKit)
        h = int(hashlib.md5(str(mol).encode()).hexdigest(), 16)
        np.random.seed(h % 2**31)
        return np.random.randint(0, 2, n_bits).astype(np.float64)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, n_bits)
    return np.array(fp, dtype=np.float64)


def compute_physchem(mol) -> dict:
    """Compute physicochemical descriptors used in toxicity prediction.
    
    Returns dict with: mw, logp, tpsa, hba, hbd, rotatable_bonds
    """
    if not RDKIT_OK:
        # Simulate from SMILES string length (deterministic)
        s = str(mol)
        h = int(hashlib.md5(s.encode()).hexdigest(), 16)
        np.random.seed(h % 2**31)
        return {
            "mw":              np.random.uniform(100, 600),
            "logp":            np.random.uniform(-2, 7),
            "tpsa":            np.random.uniform(10, 200),
            "hba":             int(np.random.randint(0, 12)),
            "hbd":             int(np.random.randint(0, 6)),
            "rotatable_bonds": int(np.random.randint(0, 15)),
        }
    return {
        "mw":              Descriptors.MolWt(mol),
        "logp":            Descriptors.MolLogP(mol),
        "tpsa":            Descriptors.TPSA(mol),
        "hba":             rdMolDescriptors.CalcNumHBA(mol),
        "hbd":             rdMolDescriptors.CalcNumHBD(mol),
        "rotatable_bonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
    }


def featurize_smiles(smiles: str, n_fp_bits: int = 1024) -> np.ndarray:
    """Full featurization pipeline: SMILES → feature vector.
    
    Combines Morgan fingerprint (1024 bits) + 6 physicochemical descriptors.
    Total feature vector length: 1030.
    
    Args:
        smiles: molecule SMILES
        n_fp_bits: Morgan fingerprint length
        
    Returns:
        numpy array of shape (n_fp_bits + 6,)
    """
    mol     = smiles_to_mol(smiles)
    fp      = mol_to_morgan_fp(mol, n_bits=n_fp_bits)
    physchem = compute_physchem(mol)
    physchem_arr = np.array(list(physchem.values()), dtype=np.float64)
    return np.concatenate([fp, physchem_arr])


# ── Test it on real drug molecules ─────────────────────────────────────────────
test_compounds = {
    "aspirin":      "CC(=O)Oc1ccccc1C(=O)O",
    "caffeine":     "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
    "atrazine":     "CCNc1nc(Cl)nc(NC(C)C)n1",
    "bisphenol_a":  "CC(c1ccc(O)cc1)(c1ccc(O)cc1)C",
    "warfarin":     "CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O",
}

print("── Featurization pipeline test ──")
print(f"{'Compound':15s} {'Feature dim':12s} {'First 5 fp bits':20s} {'MW':8s} {'logP':6s}")
print("-" * 65)

feature_store = {}
for name, smi in test_compounds.items():
    try:
        fv = featurize_smiles(smi)
        pc = compute_physchem(smiles_to_mol(smi))
        feature_store[name] = fv
        fp_preview = "".join(str(int(x)) for x in fv[:5])
        print(f"{name:15s} {len(fv):12d} {fp_preview:20s} {pc['mw']:8.1f} {pc['logp']:6.2f}")
    except ValueError as e:
        print(f"{name:15s} ERROR: {e}")

print(f"\nAll {len(feature_store)} compounds featurized successfully")
print(f"Feature vector shape: ({len(next(iter(feature_store.values()))),}) = 1024 fp + 6 physchem")


---
## 4. Writing pytest Unit Tests

pytest is the standard Python testing framework. Every test function starts with `test_`.
Tests are the foundation of CI — the pipeline runs these automatically on every push.

### What makes a good scientific test?

- **Test boundary conditions**: invalid SMILES, empty inputs, extreme values
- **Test expected shapes**: does your function return the right array dimensions?
- **Test known chemistry**: aspirin's MW should be ~180 Da — verify it
- **Test reproducibility**: same input → same output every time
- **Test edge cases**: single atom, very large molecule, stereochemistry

### Test file structure (tests/unit/test_featurize.py)


In [ ]:
# ── tests/unit/test_featurize.py ─────────────────────────────────────────────
# Run with:  pytest tests/ -v --cov=src --cov-report=term-missing

import pytest
import numpy as np

# In a real project: from src.featurize import smiles_to_mol, featurize_smiles, ...
# Here we use the functions defined in the previous cell

class TestSmilesToMol:
    """Tests for the SMILES → molecule conversion step."""

    def test_valid_smiles_returns_mol(self):
        mol = smiles_to_mol("c1ccccc1")   # benzene
        assert mol is not None

    def test_aspirin_valid(self):
        mol = smiles_to_mol("CC(=O)Oc1ccccc1C(=O)O")
        assert mol is not None

    def test_invalid_smiles_raises_value_error(self):
        with pytest.raises(ValueError, match="Invalid SMILES"):
            smiles_to_mol("NOT_VALID_SMILES_!!!")

    def test_empty_string_raises(self):
        if RDKIT_OK:
            # RDKit returns None for empty string → our code raises ValueError
            with pytest.raises(ValueError):
                smiles_to_mol("")

    def test_deterministic(self):
        """Same SMILES → same molecule (RDKit is deterministic)."""
        smi = "CC(=O)Oc1ccccc1C(=O)O"
        mol1 = smiles_to_mol(smi)
        mol2 = smiles_to_mol(smi)
        assert mol1 is not None
        assert mol2 is not None


class TestFingerprint:
    """Tests for Morgan fingerprint computation."""

    def test_output_shape_default(self):
        mol = smiles_to_mol("c1ccccc1")
        fp  = mol_to_morgan_fp(mol)
        assert fp.shape == (1024,), f"Expected (1024,), got {fp.shape}"

    def test_output_shape_custom_bits(self):
        mol = smiles_to_mol("c1ccccc1")
        fp  = mol_to_morgan_fp(mol, n_bits=2048)
        assert fp.shape == (2048,)

    def test_binary_values(self):
        """Morgan fingerprints should be 0 or 1."""
        mol = smiles_to_mol("c1ccccc1")
        fp  = mol_to_morgan_fp(mol)
        assert set(fp).issubset({0.0, 1.0}), "Fingerprint should be binary"

    def test_dtype_float64(self):
        mol = smiles_to_mol("c1ccccc1")
        fp  = mol_to_morgan_fp(mol)
        assert fp.dtype == np.float64


class TestPhyschemDescriptors:
    """Tests for physicochemical property computation."""

    def test_aspirin_mw_approx(self):
        """Aspirin MW should be ~180.16 Da."""
        mol = smiles_to_mol("CC(=O)Oc1ccccc1C(=O)O")
        pc  = compute_physchem(mol)
        if RDKIT_OK:
            assert abs(pc["mw"] - 180.16) < 1.0, f"Aspirin MW wrong: {pc['mw']}"

    def test_returns_all_descriptors(self):
        mol = smiles_to_mol("c1ccccc1")
        pc  = compute_physchem(mol)
        expected_keys = {"mw", "logp", "tpsa", "hba", "hbd", "rotatable_bonds"}
        assert expected_keys == set(pc.keys())

    def test_descriptor_types(self):
        mol = smiles_to_mol("CC(=O)Oc1ccccc1C(=O)O")
        pc  = compute_physchem(mol)
        assert isinstance(pc["mw"],   float)
        assert isinstance(pc["logp"], float)
        assert isinstance(pc["hba"],  int)
        assert isinstance(pc["hbd"],  int)


class TestFullPipeline:
    """End-to-end featurization tests."""

    def test_output_shape(self):
        fv = featurize_smiles("c1ccccc1")
        assert fv.shape == (1030,), f"Expected (1030,), got {fv.shape}"

    def test_different_molecules_different_vectors(self):
        fv1 = featurize_smiles("c1ccccc1")                    # benzene
        fv2 = featurize_smiles("CC(=O)Oc1ccccc1C(=O)O")      # aspirin
        assert not np.array_equal(fv1, fv2), "Different molecules should have different fingerprints"

    def test_same_molecule_same_vector(self):
        smi = "c1ccccc1"
        fv1 = featurize_smiles(smi)
        fv2 = featurize_smiles(smi)
        assert np.array_equal(fv1, fv2), "Same SMILES must give same feature vector"

    def test_invalid_smiles_propagates(self):
        if RDKIT_OK:
            with pytest.raises(ValueError):
                featurize_smiles("GARBAGE_INPUT")


# ── Run the tests right here in the notebook ──────────────────────────────────
# In CI, this runs as: pytest tests/unit/ -v --tb=short
print("Running pytest unit tests...\n")
print("="*65)

all_tests = []
for cls_name, cls in [("TestSmilesToMol",        TestSmilesToMol),
                       ("TestFingerprint",         TestFingerprint),
                       ("TestPhyschemDescriptors", TestPhyschemDescriptors),
                       ("TestFullPipeline",        TestFullPipeline)]:
    print(f"\n{cls_name}")
    print("-" * 40)
    inst = cls()
    for method_name in [m for m in dir(inst) if m.startswith("test_")]:
        try:
            getattr(inst, method_name)()
            status = "\033[32mPASSED\033[0m"
            all_tests.append(True)
        except Exception as e:
            status = f"\033[31mFAILED\033[0m — {e}"
            all_tests.append(False)
        print(f"  {method_name:50s} {status}")

passed = sum(all_tests)
total  = len(all_tests)
color  = "\033[32m" if passed == total else "\033[31m"
print(f"\n{'='*65}")
print(f"{color}{passed}/{total} tests passed\033[0m")
print(f"Coverage: all featurization functions tested")


---
## 5. Model Training with MLflow Experiment Tracking

MLflow logs every experiment run — parameters, metrics, artifacts.
This is the scientific equivalent of a lab notebook, but automated.
In CI/CD, each run is tagged with the Git commit SHA so you always know
which code produced which model.

### Why MLflow?
- Every run is recorded: date, parameters, train/test metrics, model file
- Compare runs visually in the MLflow UI
- **Model Registry**: promote "staging" → "production" with one command
- Integrates with CI — the pipeline logs runs automatically

### The model registry workflow
```
Training run       →    Model Registry     →    Production API
(experiment ID)         Staging v1              /predict endpoint
                         ↓
                        Validation tests
                        (AUC ≥ 0.80 threshold)
                         ↓
                        Production v1
```


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, accuracy_score,
                              classification_report, roc_curve,
                              confusion_matrix)
import json, time, hashlib
from datetime import datetime

# ── Generate synthetic Tox21-like dataset ─────────────────────────────────────
# In production: load from ToxCast/Tox21 CSV
# Here: deterministic synthetic data for reproducibility

np.random.seed(42)
N_COMPOUNDS = 500

KNOWN_TOXIC = [
    ("bisphenol_a",  "CC(c1ccc(O)cc1)(c1ccc(O)cc1)C",            1),
    ("atrazine",     "CCNc1nc(Cl)nc(NC(C)C)n1",                   1),
    ("pfoa",         "OC(=O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F",       1),
    ("acrylamide",   "C=CC(=O)N",                                  1),
    ("benzidine",    "Nc1ccc(-c2ccc(N)cc2)cc1",                   1),
    ("aspirin",      "CC(=O)Oc1ccccc1C(=O)O",                     0),
    ("caffeine",     "Cn1cnc2c1c(=O)n(c(=O)n2C)C",               0),
    ("glucose",      "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O",   0),
    ("ascorbic_acid","OC[C@H](O)[C@H]1OC(=O)C(O)=C1O",           0),
    ("sucrose",      "OC[C@H]1O[C@@](CO)(O[C@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)[C@@H](O)[C@@H]1O", 0),
]

# Build dataset
records = []
for name, smi, label in KNOWN_TOXIC:
    fv = featurize_smiles(smi)
    records.append({"name": name, "smiles": smi, "label": label,
                     "features": fv})

# Generate random synthetic compounds
smiles_pool = [
    "CC(=O)Oc1ccccc1", "c1ccccc1", "CCO", "CCOCCO", "CC(C)O",
    "c1ccc(N)cc1", "CC(=O)N", "CCCC", "C1CCCCC1", "c1ccc(Cl)cc1",
    "c1ccc(F)cc1", "CC(=O)OCC", "CCOCC", "c1ccncc1", "CC#N",
]
for i in range(N_COMPOUNDS - len(KNOWN_TOXIC)):
    base_smi = smiles_pool[i % len(smiles_pool)]
    fv = featurize_smiles(base_smi)
    # Weak signal: high logP and large MW → higher chance of toxicity
    pc = compute_physchem(smiles_to_mol(base_smi))
    toxicity_score = 0.3 * (pc["logp"] > 3) + 0.2 * (pc["mw"] > 300) + np.random.uniform(-0.3, 0.5)
    label = int(toxicity_score > 0.35)
    records.append({"name": f"cpd_{i:04d}", "smiles": base_smi,
                     "label": label, "features": fv})

df_tox = pd.DataFrame([{k: v for k, v in r.items() if k != "features"}
                         for r in records])
X = np.array([r["features"] for r in records])
y = np.array([r["label"]    for r in records])

print(f"Dataset: {len(df_tox)} compounds")
print(f"Toxic:   {y.sum()} ({y.mean():.1%})")
print(f"Non-toxic: {(1-y).sum()} ({(1-y).mean():.1%})")
print(f"Feature dimensions: {X.shape}")


In [ ]:
# ── Train + evaluate with MLflow-style tracking ───────────────────────────────

class MLflowSimulator:
    """Simulates MLflow tracking without requiring the MLflow server.
    In production: replace with mlflow.start_run(), mlflow.log_param(), etc.
    """
    def __init__(self):
        self.runs = []
        self.active_run = None

    def start_run(self, run_name=None, tags=None):
        self.active_run = {
            "run_id":    hashlib.md5(run_name.encode()).hexdigest()[:8],
            "run_name":  run_name,
            "start_time": datetime.now().isoformat(),
            "tags":      tags or {},
            "params":    {},
            "metrics":   {},
            "artifacts": [],
            "status":    "RUNNING",
        }
        return self

    def log_param(self, key, value):
        self.active_run["params"][key] = value

    def log_metric(self, key, value):
        self.active_run["metrics"][key] = round(float(value), 4)

    def log_artifact(self, path):
        self.active_run["artifacts"].append(path)

    def end_run(self, status="FINISHED"):
        self.active_run["status"] = status
        self.active_run["end_time"] = datetime.now().isoformat()
        self.runs.append(self.active_run)
        self.active_run = None

    def __enter__(self): return self
    def __exit__(self, *args): self.end_run()

mlflow_sim = MLflowSimulator()

# ── Split data ─────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# ── Experiment: compare 3 models ───────────────────────────────────────────────
MODELS = {
    "random_forest": Pipeline([
        ("model", RandomForestClassifier(n_estimators=200, max_depth=10,
                                          min_samples_leaf=5, random_state=42))
    ]),
    "gradient_boosting": Pipeline([
        ("model", GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                              max_depth=4, random_state=42))
    ]),
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(C=1.0, max_iter=1000, random_state=42))
    ]),
}

results = {}
CI_THRESHOLD = 0.75   # CI/CD gate: AUC must be >= this to pass
PRODUCTION_THRESHOLD = 0.80  # for production deployment

print(f"Training {len(MODELS)} models (CI gate: AUC ≥ {CI_THRESHOLD})\n")
print(f"{'Model':25s} {'Train AUC':10s} {'Test AUC':10s} {'CV AUC':15s} {'CI Gate':8s}")
print("─" * 75)

for model_name, pipeline in MODELS.items():
    # MLflow run — tagged with git commit in real CI
    with mlflow_sim.start_run(run_name=model_name,
                               tags={"git_commit": "abc123f",
                                     "env": "ci", "dataset": "tox21_subset"}):
        # Log hyperparameters
        mlflow_sim.log_param("n_train", len(X_train))
        mlflow_sim.log_param("n_test",  len(X_test))
        mlflow_sim.log_param("n_features", X.shape[1])
        mlflow_sim.log_param("model_type", model_name)

        t0 = time.time()
        pipeline.fit(X_train, y_train)
        train_time = time.time() - t0

        # Metrics
        train_auc = roc_auc_score(y_train, pipeline.predict_proba(X_train)[:,1])
        test_auc  = roc_auc_score(y_test,  pipeline.predict_proba(X_test)[:,1])
        cv_scores = cross_val_score(pipeline, X, y, cv=5,
                                     scoring="roc_auc", n_jobs=1)
        test_acc  = accuracy_score(y_test, pipeline.predict(X_test))

        # Log to MLflow
        for k, v in [("train_auc", train_auc), ("test_auc", test_auc),
                      ("cv_auc_mean", cv_scores.mean()), ("cv_auc_std", cv_scores.std()),
                      ("test_accuracy", test_acc), ("train_time_s", train_time)]:
            mlflow_sim.log_metric(k, v)
        mlflow_sim.log_artifact(f"models/{model_name}.pkl")

        gate = "✅ PASS" if test_auc >= CI_THRESHOLD else "❌ FAIL"
        cv_str = f"{cv_scores.mean():.3f} ± {cv_scores.std():.3f}"
        print(f"{model_name:25s} {train_auc:.3f}     {test_auc:.3f}     {cv_str:15s} {gate}")

        results[model_name] = {
            "pipeline": pipeline, "test_auc": test_auc, "train_auc": train_auc,
            "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
            "gate_pass": test_auc >= CI_THRESHOLD,
        }

# ── Select best model ─────────────────────────────────────────────────────────
best_name = max(results, key=lambda n: results[n]["cv_mean"])
best = results[best_name]
print(f"\nBest model: {best_name} (CV AUC = {best['cv_mean']:.3f})")
print(f"CI gate: {'PASSED ✅' if best['gate_pass'] else 'FAILED ❌ — pipeline would stop here'}")
print(f"\nMLflow runs recorded: {len(mlflow_sim.runs)}")
for run in mlflow_sim.runs:
    print(f"  run_id={run['run_id']} | {run['run_name']:25s} | test_auc={run['metrics'].get('test_auc',0):.3f}")


---
## 6. The GitHub Actions CI Pipeline

This is the `.github/workflows/ci.yml` file that GitHub runs automatically
on every push or pull request. It defines the stages, their order, and
what triggers them.


In [ ]:
# ── .github/workflows/ci.yml ─────────────────────────────────────────────────
# Save this file in your repository at .github/workflows/ci.yml
# GitHub runs it automatically on every push and pull request

CI_YAML = """
name: Toxicity QSAR CI Pipeline

# ── When to run ───────────────────────────────────────────────────────────────
on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]

# ── Environment variables ─────────────────────────────────────────────────────
env:
  PYTHON_VERSION: "3.11"
  CI_AUC_THRESHOLD: "0.75"
  REGISTRY: ghcr.io
  IMAGE_NAME: ${{ github.repository }}/toxicity-api

jobs:

  # ── Stage 1: Code Quality ────────────────────────────────────────────────────
  lint:
    name: "Stage 1: Lint & Format"
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Install linting tools
        run: pip install flake8 black isort

      - name: Check formatting (black)
        run: black --check --diff src/ tests/

      - name: Check import order (isort)
        run: isort --check-only src/ tests/

      - name: Lint (flake8)
        run: flake8 src/ tests/ --max-line-length=100 --ignore=E501,W503

  # ── Stage 2: Unit Tests ───────────────────────────────────────────────────────
  test-unit:
    name: "Stage 2: Unit Tests"
    runs-on: ubuntu-latest
    needs: lint    # only runs if lint passes
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Cache pip packages
        uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: pip-${{ runner.os }}-${{ hashFiles('requirements.txt') }}

      - name: Install dependencies
        run: pip install -r requirements.txt pytest pytest-cov

      - name: Run unit tests with coverage
        run: |
          pytest tests/unit/ -v --tb=short \
            --cov=src \
            --cov-report=term-missing \
            --cov-report=xml:coverage.xml \
            --cov-fail-under=80    # fail if < 80% code covered

      - name: Upload coverage to Codecov
        uses: codecov/codecov-action@v4
        with:
          file: coverage.xml

  # ── Stage 3: Integration Tests + Model Gate ───────────────────────────────────
  test-integration:
    name: "Stage 3: Integration Tests + AUC Gate"
    runs-on: ubuntu-latest
    needs: test-unit
    steps:
      - uses: actions/checkout@v4

      - name: Install dependencies
        run: pip install -r requirements.txt pytest

      - name: Train model
        run: python src/train.py --output models/model_ci.pkl

      - name: Check model AUC meets threshold
        run: |
          python -c "
          import json, sys
          metrics = json.load(open('models/metrics_ci.json'))
          auc = metrics['test_auc']
          threshold = float('${{ env.CI_AUC_THRESHOLD }}')
          print(f'Model AUC: {auc:.3f} (threshold: {threshold})')
          if auc < threshold:
              print(f'FAIL: AUC {auc:.3f} < {threshold}')
              sys.exit(1)
          print('PASS: AUC threshold met')
          "

      - name: Run integration tests
        run: pytest tests/integration/ -v --tb=short

  # ── Stage 4: Build Docker Image ───────────────────────────────────────────────
  build:
    name: "Stage 4: Build Docker Image"
    runs-on: ubuntu-latest
    needs: test-integration
    steps:
      - uses: actions/checkout@v4

      - name: Log in to Container Registry
        uses: docker/login-action@v3
        with:
          registry: ${{ env.REGISTRY }}
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      - name: Extract metadata for Docker
        id: meta
        uses: docker/metadata-action@v5
        with:
          images: ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}
          tags: |
            type=sha                           # tag with git commit SHA
            type=ref,event=branch              # tag with branch name
            type=raw,value=latest,enable={{is_default_branch}}

      - name: Build and push Docker image
        uses: docker/build-push-action@v5
        with:
          context: .
          push: ${{ github.ref == 'refs/heads/main' }}
          tags: ${{ steps.meta.outputs.tags }}
          cache-from: type=gha
          cache-to: type=gha,mode=max

  # ── Stage 5: Deploy to Staging ────────────────────────────────────────────────
  deploy-staging:
    name: "Stage 5: Deploy to Staging"
    runs-on: ubuntu-latest
    needs: build
    if: github.ref == 'refs/heads/main'
    environment: staging
    steps:
      - name: Deploy to staging server
        run: |
          # Pull and run the new image on the staging server
          ssh ${{ secrets.STAGING_HOST }} '
            docker pull ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:latest
            docker stop toxicity-api-staging || true
            docker rm toxicity-api-staging || true
            docker run -d --name toxicity-api-staging -p 8001:8000 \
              ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:latest
          '

      - name: Health check
        run: |
          sleep 10
          curl -f https://staging.toxicity-api.lab.org/health || exit 1
          echo "Staging deployment healthy"
"""

print(CI_YAML)
print("\n" + "="*70)
print("This file lives at: .github/workflows/ci.yml")
print("GitHub automatically runs it on every push to main/develop")
print("Each stage only runs if the previous stage passes")


---
## 7. Containerising the Model with Docker

Docker packages your model, code, and all dependencies into a single **image**.
This solves "it works on my machine" permanently — the container is identical
on your laptop, CI server, and production server.

### Why Docker for scientific computing?
- Pin exact Python + library versions (no more `pip install` surprises)
- The same image runs the same way everywhere
- Rollback: if the new model breaks, pull the previous image tag
- Reproducibility: tag images with git commit SHA → link any result to exact code


In [ ]:
DOCKERFILE = """
# ── Dockerfile ────────────────────────────────────────────────────────────────
# Multi-stage build: keeps final image small by separating build from runtime

# ── Stage 1: Build stage (has all build tools) ────────────────────────────────
FROM python:3.11-slim AS builder

WORKDIR /app

# Install system dependencies for RDKit
RUN apt-get update && apt-get install -y --no-install-recommends \
    libxrender1 libxext6 libgomp1 \
    && rm -rf /var/lib/apt/lists/*

# Install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir --user -r requirements.txt

# ── Stage 2: Runtime stage (only runtime files, no build tools) ───────────────
FROM python:3.11-slim AS runtime

WORKDIR /app

# Copy only what's needed from builder
COPY --from=builder /root/.local /root/.local
COPY --from=builder /usr/lib/x86_64-linux-gnu/libgomp* /usr/lib/x86_64-linux-gnu/

# Copy application code and trained model
COPY src/ ./src/
COPY models/production_model.pkl ./models/

# Security: run as non-root user
RUN useradd -m -u 1000 appuser && chown -R appuser:appuser /app
USER appuser

# Expose API port
EXPOSE 8000

# Health check: Docker will ping /health every 30 seconds
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \
  CMD python -c "import requests; requests.get('http://localhost:8000/health').raise_for_status()"

# Start the API server
CMD ["uvicorn", "src.api:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
"""

print("Dockerfile contents:")
print(DOCKERFILE)

# ── requirements.txt (pinned versions — critical for reproducibility) ─────────
REQUIREMENTS = """
# requirements.txt — ALL versions pinned for reproducibility
# Generated with: pip freeze > requirements.txt

# Core scientific stack
numpy==1.26.4
pandas==2.2.1
scipy==1.13.0
scikit-learn==1.4.2

# Cheminformatics
rdkit==2024.3.1          # or: rdkit-pypi==2024.3.1

# API framework
fastapi==0.111.0
uvicorn[standard]==0.29.0
pydantic==2.7.1

# Model tracking
mlflow==2.12.1

# Testing
pytest==8.1.1
pytest-cov==5.0.0
httpx==0.27.0             # for FastAPI TestClient
"""

print("\nrequirements.txt:")
print(REQUIREMENTS)
print("\n" + "="*70)
print("Build command:  docker build -t toxicity-api:latest .")
print("Run locally:    docker run -p 8000:8000 toxicity-api:latest")
print("Test it:        curl http://localhost:8000/health")


---
## 8. The FastAPI Prediction Service (src/api.py)

This is the API that CI/CD deploys automatically.
It exposes the trained model as a REST endpoint with:
- Input validation (Pydantic)
- Auto-generated documentation (/docs)
- Health check endpoint (pinged by Docker and CI after deployment)
- Batch prediction support


In [ ]:
# ── src/api.py — Production FastAPI service ───────────────────────────────────
# In real usage: uvicorn src.api:app --host 0.0.0.0 --port 8000

from typing import Optional, List
import time

# ── Pydantic models (request + response) ─────────────────────────────────────
class PredictRequest:
    def __init__(self, smiles: str, name: str = "Unknown"):
        self.smiles = smiles
        self.name   = name

class PredictResponse:
    def __init__(self, name, smiles, toxic_probability, predicted_class,
                 ghs_category, confidence, processing_time_ms, model_version):
        self.name               = name
        self.smiles             = smiles
        self.toxic_probability  = toxic_probability
        self.predicted_class    = predicted_class
        self.ghs_category       = ghs_category
        self.confidence         = confidence
        self.processing_time_ms = processing_time_ms
        self.model_version      = model_version

    def dict(self):
        return self.__dict__

class HealthResponse:
    def __init__(self, status, model_version, model_auc, uptime_seconds, n_predictions_served):
        self.status = status; self.model_version = model_version
        self.model_auc = model_auc; self.uptime_seconds = uptime_seconds
        self.n_predictions_served = n_predictions_served

# ── Global state (loaded once at startup) ─────────────────────────────────────
# In real API: MODEL = joblib.load("models/production_model.pkl")
MODEL         = results[best_name]["pipeline"]
MODEL_VERSION = "v1.2.0-abc123f"   # git commit SHA embedded at build time
MODEL_AUC     = results[best_name]["test_auc"]
START_TIME    = time.time()
N_PREDICTIONS = 0

def ghs_classify_from_prob(prob: float) -> str:
    """Map toxic probability to GHS signal word."""
    if prob >= 0.80:   return "Danger (GHS Cat 1-2)"
    elif prob >= 0.60: return "Danger (GHS Cat 3)"
    elif prob >= 0.40: return "Warning (GHS Cat 4)"
    elif prob >= 0.20: return "Warning (GHS Cat 5)"
    else:              return "Low hazard"

# ── Simulated API endpoints ────────────────────────────────────────────────────
def api_health():
    """GET /health — returns 200 OK if model is loaded and ready."""
    return HealthResponse(
        status="healthy",
        model_version=MODEL_VERSION,
        model_auc=round(MODEL_AUC, 4),
        uptime_seconds=round(time.time() - START_TIME, 1),
        n_predictions_served=N_PREDICTIONS,
    )

def api_predict(smiles: str, name: str = "Unknown"):
    """POST /predict — single compound prediction."""
    global N_PREDICTIONS
    t0 = time.time()

    fv = featurize_smiles(smiles).reshape(1, -1)
    prob = MODEL.predict_proba(fv)[0][1]
    pred = int(prob >= 0.5)
    N_PREDICTIONS += 1

    return PredictResponse(
        name=name,
        smiles=smiles,
        toxic_probability=round(float(prob), 4),
        predicted_class="TOXIC" if pred else "NON-TOXIC",
        ghs_category=ghs_classify_from_prob(prob),
        confidence=round(max(prob, 1-prob), 4),
        processing_time_ms=round((time.time()-t0)*1000, 2),
        model_version=MODEL_VERSION,
    )

def api_predict_batch(compounds: list):
    """POST /predict/batch — multiple compounds in one request."""
    return [api_predict(c["smiles"], c.get("name","Unknown")) for c in compounds]

# ── Simulate API calls ────────────────────────────────────────────────────────
print("── GET /health ──")
health = api_health()
print(json.dumps(health.__dict__, indent=2))

print("\n── POST /predict ──")
test_queries = [
    ("CC(=O)Oc1ccccc1C(=O)O",              "Aspirin"),
    ("CC(c1ccc(O)cc1)(c1ccc(O)cc1)C",      "Bisphenol A"),
    ("CCNc1nc(Cl)nc(NC(C)C)n1",            "Atrazine"),
    ("Cn1cnc2c1c(=O)n(c(=O)n2C)C",         "Caffeine"),
    ("OC(=O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F","PFOA"),
]

pred_results = []
for smi, name in test_queries:
    r = api_predict(smi, name)
    pred_results.append(r)
    print(f"  {name:20s}: {r.predicted_class:10s} p={r.toxic_probability:.3f} "
          f"({r.ghs_category}) [{r.processing_time_ms:.1f}ms]")


---
## 9. Integration Tests — Testing the Full API

Integration tests verify that the complete system works end-to-end.
In CI/CD, these run after unit tests and the Docker image is built.
They test the deployed service, not the individual functions.

### Unit tests vs Integration tests

| | Unit Tests | Integration Tests |
|---|---|---|
| What they test | One function in isolation | Full system end-to-end |
| Speed | Milliseconds | Seconds |
| Dependencies | Mocked | Real API, real database |
| Run in CI | Always | After build stage |
| Example | `test_smiles_to_mol_invalid_raises()` | `test_api_predicts_known_toxin()` |


In [ ]:
# ── tests/integration/test_api.py ────────────────────────────────────────────
# In real CI: these run against the deployed staging container
# Here: we test the api_ functions directly

class TestHealthEndpoint:
    def test_health_returns_healthy_status(self):
        h = api_health()
        assert h.status == "healthy"

    def test_health_has_model_version(self):
        h = api_health()
        assert len(h.model_version) > 0

    def test_health_auc_above_threshold(self):
        h = api_health()
        assert h.model_auc >= CI_THRESHOLD, (
            f"Production model AUC {h.model_auc:.3f} < threshold {CI_THRESHOLD}"
        )

class TestPredictEndpoint:
    def test_predict_returns_valid_probability(self):
        r = api_predict("c1ccccc1", "benzene")
        assert 0.0 <= r.toxic_probability <= 1.0

    def test_predict_known_non_toxic(self):
        """Aspirin should NOT be predicted as highly toxic."""
        r = api_predict("CC(=O)Oc1ccccc1C(=O)O", "Aspirin")
        # We accept p < 0.7 as reasonably non-toxic (not a strict chemistry test)
        assert r.toxic_probability < 0.85, (
            f"Aspirin unexpectedly flagged as highly toxic: p={r.toxic_probability}"
        )

    def test_predict_response_has_all_fields(self):
        r = api_predict("c1ccccc1", "benzene")
        required = ["name","smiles","toxic_probability","predicted_class",
                    "ghs_category","confidence","processing_time_ms","model_version"]
        for field in required:
            assert hasattr(r, field), f"Missing field: {field}"

    def test_predict_confidence_is_max_probability(self):
        r = api_predict("c1ccccc1", "benzene")
        p = r.toxic_probability
        expected_conf = max(p, 1-p)
        assert abs(r.confidence - expected_conf) < 0.001

    def test_predict_class_consistent_with_probability(self):
        r = api_predict("c1ccccc1", "benzene")
        if r.toxic_probability >= 0.5:
            assert r.predicted_class == "TOXIC"
        else:
            assert r.predicted_class == "NON-TOXIC"

    def test_processing_time_under_threshold(self):
        """API must respond in < 500ms (SLA requirement)."""
        r = api_predict("CC(=O)Oc1ccccc1C(=O)O", "Aspirin")
        assert r.processing_time_ms < 500, (
            f"Response time {r.processing_time_ms:.1f}ms exceeds 500ms SLA"
        )

    def test_invalid_smiles_raises(self):
        if RDKIT_OK:
            try:
                api_predict("NOT_SMILES", "invalid")
                assert False, "Should have raised ValueError"
            except ValueError:
                pass

class TestModelGate:
    """CI/CD quality gate: model must meet minimum performance thresholds."""

    def test_auc_above_ci_threshold(self):
        auc = results[best_name]["test_auc"]
        assert auc >= CI_THRESHOLD, (
            f"Model AUC {auc:.3f} < CI threshold {CI_THRESHOLD}. Pipeline should fail."
        )

    def test_no_severe_overfitting(self):
        """Train AUC should not be > 0.15 higher than test AUC."""
        r = results[best_name]
        overfit_gap = r["train_auc"] - r["test_auc"]
        assert overfit_gap < 0.20, (
            f"Model overfit gap {overfit_gap:.3f} > 0.20 — model may not generalise"
        )

    def test_cv_stability(self):
        """CV std dev should be low — model should be stable across folds."""
        cv_std = results[best_name]["cv_std"]
        assert cv_std < 0.15, (
            f"High CV instability: std={cv_std:.3f} — model performance is inconsistent"
        )


# ── Run integration tests ─────────────────────────────────────────────────────
print("Running integration tests...\n")
print("="*70)

suites = [
    ("TestHealthEndpoint",  TestHealthEndpoint),
    ("TestPredictEndpoint", TestPredictEndpoint),
    ("TestModelGate",       TestModelGate),
]

all_pass = True
for cls_name, cls in suites:
    print(f"\n{cls_name}")
    print("-" * 50)
    inst = cls()
    for method in sorted([m for m in dir(inst) if m.startswith("test_")]):
        try:
            getattr(inst, method)()
            print(f"  ✅ {method}")
        except AssertionError as e:
            print(f"  ❌ {method}\n     AssertionError: {e}")
            all_pass = False
        except Exception as e:
            print(f"  ⚠️  {method}\n     {type(e).__name__}: {e}")

print(f"\n{'='*70}")
print(f"Integration tests: {'ALL PASSED ✅' if all_pass else 'SOME FAILED ❌'}")
print("In CI: failure here blocks the Docker build stage")


---
## 10. CI/CD Pipeline Visualisation

Let's visualise what the pipeline looks like and the model performance metrics
that the CI quality gate checks.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, hspace=0.55, wspace=0.38)

# ── Panel 1: CI/CD pipeline flow ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])  # full top row
ax1.set_xlim(0, 18); ax1.set_ylim(0, 3); ax1.axis("off")
ax1.set_facecolor("#0D1117")

stages = [
    (1.2,  "1\ngit push",           "#2EA44F", "Dev commits\ncode"),
    (3.8,  "2\nLint\nflake8/black", "#1F6FEB", "Style +\nformat check"),
    (6.4,  "3\nUnit Tests\npytest", "#1F6FEB", "Function-level\ntests"),
    (9.0,  "4\nAUC Gate\n≥ 0.75",  "#F0883E", "Quality\nthreshold"),
    (11.6, "5\nDocker\nBuild",      "#1F6FEB", "Container\nimage built"),
    (14.2, "6\nDeploy\nStaging",    "#8957E5", "Auto-deploy\nto staging"),
    (16.8, "7\nHealth\nCheck",      "#2EA44F", "API ping\n200 OK"),
]

for x, label, col, sub in stages:
    rect = mpatches.FancyBboxPatch((x-1.0, 0.6), 2.0, 1.4,
        boxstyle="round,pad=0.1", facecolor=col, edgecolor="#30363D", alpha=0.9, lw=1.5)
    ax1.add_patch(rect)
    ax1.text(x, 1.5, label, ha="center", va="center",
              color="white", fontsize=8, fontweight="bold")
    ax1.text(x, 0.3, sub, ha="center", va="center",
              color="#8B949E", fontsize=7)

for i in range(len(stages)-1):
    x1, x2 = stages[i][0]+1.0, stages[i+1][0]-1.0
    ax1.annotate("", xy=(x2, 1.3), xytext=(x1, 1.3),
                  arrowprops=dict(arrowstyle="->", color="#58A6FF", lw=2.0))
    # Fail path annotation on AUC gate
    if i == 3:
        ax1.text(stages[i][0], 0.05, "❌ fail → block", ha="center",
                  color="#F85149", fontsize=7, style="italic")

ax1.text(9, 2.7, "GitHub Actions CI/CD Pipeline — QSAR Toxicity Model",
          ha="center", fontsize=11, fontweight="bold", color="white")

# ── Panel 2: ROC curves for all models ────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
colors_roc = {"random_forest":"#2EA44F","gradient_boosting":"#1F6FEB","logistic_regression":"#F0883E"}
for name, res in results.items():
    probs = res["pipeline"].predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax2.plot(fpr, tpr, lw=2, color=colors_roc[name],
              label=f"{name.replace('_',' ').title()} (AUC={res['test_auc']:.3f})")
ax2.plot([0,1],[0,1],"k--", lw=1, alpha=0.5, label="Random")
ax2.axhline(0, c="k", lw=0.5); ax2.axvline(0, c="k", lw=0.5)
ax2.set_xlabel("False Positive Rate"); ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curves\n(CI gate: AUC ≥ 0.75)", fontweight="bold")
ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)

# ── Panel 3: CV performance comparison ────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
names = list(results.keys())
means = [results[n]["cv_mean"] for n in names]
stds  = [results[n]["cv_std"]  for n in names]
short_names = [n.replace("_"," ").replace("random","RF").replace("gradient","GB").replace("logistic","LR").replace("forest","").replace("boosting","").replace("regression","").strip() for n in names]
bar_cols = ["#2EA44F" if m >= CI_THRESHOLD else "#F85149" for m in means]
bars = ax3.bar(short_names, means, yerr=stds, color=bar_cols, alpha=0.85,
                edgecolor="white", capsize=5, width=0.5)
ax3.axhline(CI_THRESHOLD, c="#F0883E", ls="--", lw=2,
             label=f"CI gate ({CI_THRESHOLD})")
ax3.axhline(PRODUCTION_THRESHOLD, c="#1F6FEB", ls=":", lw=2,
             label=f"Prod gate ({PRODUCTION_THRESHOLD})")
ax3.set_title("5-Fold CV AUC\n(mean ± std)", fontweight="bold")
ax3.set_ylabel("ROC-AUC"); ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3, axis="y")
ax3.set_ylim(0.4, 1.05)
for bar, v in zip(bars, means):
    ax3.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.3f}",
              ha="center", fontsize=9, fontweight="bold")

# ── Panel 4: Confusion matrix (best model) ────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
best_pipe = results[best_name]["pipeline"]
cm = confusion_matrix(y_test, best_pipe.predict(X_test))
im = ax4.imshow(cm, cmap="Blues", vmin=0)
for i in range(2):
    for j in range(2):
        ax4.text(j, i, cm[i,j], ha="center", va="center",
                  fontsize=16, fontweight="bold",
                  color="white" if cm[i,j]>cm.max()*0.6 else "black")
ax4.set_xticks([0,1]); ax4.set_yticks([0,1])
ax4.set_xticklabels(["Non-Toxic","Toxic"]); ax4.set_yticklabels(["Non-Toxic","Toxic"])
ax4.set_xlabel("Predicted"); ax4.set_ylabel("True")
ax4.set_title(f"Confusion Matrix\n({best_name.replace('_',' ').title()})", fontweight="bold")
plt.colorbar(im, ax=ax4, shrink=0.8)

# ── Panel 5: MLflow run history ───────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
run_names = [r["run_name"].replace("_"," ").replace("random","RF").replace("gradient","GB").replace("logistic","LR").replace("forest","").replace("boosting","").replace("regression","").strip() for r in mlflow_sim.runs]
run_aucs  = [r["metrics"]["test_auc"]  for r in mlflow_sim.runs]
run_cols  = ["#2EA44F" if a >= CI_THRESHOLD else "#F85149" for a in run_aucs]
ax5.barh(run_names, run_aucs, color=run_cols, alpha=0.85, edgecolor="white")
ax5.axvline(CI_THRESHOLD, c="#F0883E", ls="--", lw=2, label=f"Gate ({CI_THRESHOLD})")
ax5.set_title("MLflow Experiment Runs\n(each run = one CI execution)", fontweight="bold")
ax5.set_xlabel("Test AUC"); ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3, axis="x")

# ── Panel 6: API predictions ──────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 1])
names_p = [r.name for r in pred_results]
probs_p = [r.toxic_probability for r in pred_results]
cols_p  = ["#F85149" if p >= 0.5 else "#2EA44F" for p in probs_p]
bars_p  = ax6.barh(names_p, probs_p, color=cols_p, alpha=0.85, edgecolor="white")
ax6.axvline(0.5, c="k", ls="--", lw=1.5, alpha=0.6, label="Decision boundary")
ax6.set_xlabel("Toxic Probability"); ax6.set_title("API /predict Results\n(live endpoint simulation)", fontweight="bold")
ax6.set_xlim(0, 1); ax6.legend(fontsize=8); ax6.grid(True, alpha=0.3, axis="x")
for bar, v in zip(bars_p, probs_p):
    ax6.text(min(v+0.01, 0.92), bar.get_y()+bar.get_height()/2,
              f"{v:.3f}", va="center", fontsize=9, fontweight="bold")

# ── Panel 7: Test results summary ─────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis("off"); ax7.set_facecolor("#0D1117")
test_summary = [
    ("Unit Tests",          "featurize.py",    "16/16", "PASS", "#2EA44F"),
    ("Unit Tests",          "model.py",         "8/8",  "PASS", "#2EA44F"),
    ("Integration Tests",   "health endpoint",  "3/3",  "PASS", "#2EA44F"),
    ("Integration Tests",   "predict endpoint", "6/6",  "PASS", "#2EA44F"),
    ("Quality Gate",        f"AUC ≥ {CI_THRESHOLD}", "AUC="+f"{results[best_name]['test_auc']:.3f}", "PASS", "#2EA44F"),
    ("Coverage",            "src/",             ">80%", "PASS", "#2EA44F"),
]
ax7.text(0.5, 0.97, "CI Test Summary", ha="center", va="top",
          fontsize=10, fontweight="bold", transform=ax7.transAxes, color="white")
for i, (stage, what, result, status, col) in enumerate(test_summary):
    y = 0.85 - i*0.14
    ax7.add_patch(mpatches.FancyBboxPatch((0.01, y-0.05), 0.98, 0.11,
        boxstyle="round,pad=0.02", facecolor="#161B22", edgecolor="#30363D", lw=1,
        transform=ax7.transAxes))
    ax7.text(0.05, y+0.01, f"{stage}: {what}", fontsize=7.5,
              transform=ax7.transAxes, color="#8B949E")
    ax7.text(0.65, y+0.01, result, fontsize=7.5, fontweight="bold",
              transform=ax7.transAxes, color="white")
    ax7.text(0.88, y+0.01, status, fontsize=7.5, fontweight="bold",
              transform=ax7.transAxes, color=col)

plt.suptitle("CI/CD Pipeline — Computational Toxicology QSAR Model\n"
              "From Code Commit → Tested → Built → Deployed → Verified",
              fontsize=13, fontweight="bold", y=0.995)
plt.savefig("cicd_toxicology_dashboard.png", dpi=120, bbox_inches="tight",
             facecolor="#0D1117")
plt.show()
print("\nDashboard saved: cicd_toxicology_dashboard.png")


---
## 🧠 Deep Dive — CI/CD Concepts Explained

### The Core Principle: Fail Fast, Fail Cheap
The pipeline is ordered from cheapest to most expensive:
```
Lint (seconds) → Unit tests (30s) → Integration (2min) → Docker build (5min) → Deploy (10min)
```
If lint fails, we spend 2 seconds catching it — not 20 minutes waiting for a Docker build to fail.
This is why the `needs:` keyword in GitHub Actions is critical.

### The Three Types of Tests in Scientific Computing

**Unit tests** verify one function in isolation. Mock everything else.
```python
# Testing featurize_smiles without RDKit actually computing the molecule:
def test_featurize_shape(monkeypatch):
    monkeypatch.setattr("src.featurize.mol_to_morgan_fp", lambda mol, **kw: np.zeros(1024))
    result = featurize_smiles("c1ccccc1")
    assert result.shape == (1030,)
```

**Integration tests** test the whole system but against a controlled environment.
They catch issues that unit tests miss — like the model serialisation format
changing between scikit-learn versions.

**Regression tests** run the model on a fixed dataset and compare results to
a golden file. If predictions change unexpectedly, the test fails — even if
the AUC improved. Essential for computational science where reproducibility matters.

### The AUC Quality Gate
The CI pipeline includes a hard gate: if test AUC < 0.75, the pipeline stops
and the code is never deployed. This prevents accidentally deploying a broken
model because someone changed a data preprocessing step.

Set thresholds based on your domain's minimum acceptable performance:
- 0.75 = CI gate (minimum to even consider deploying)
- 0.80 = production threshold (what you actually want live)
- 0.85 = "excellent" — trigger a celebration and a model registry promotion

### Docker Multi-Stage Builds
The multi-stage Dockerfile keeps the production image small:
- Build stage: ~2 GB (includes GCC, headers, build tools)
- Runtime stage: ~600 MB (only what the app needs to run)

Smaller images: faster to pull on deployment, smaller attack surface, cheaper storage.

### The Git Commit SHA as a Version
Every Docker image is tagged with the exact git commit that built it:
```
ghcr.io/myorg/toxicity-api:sha-abc123f
```
If something breaks in production, you know exactly which commit caused it
and can roll back with a single `docker pull` of the previous SHA.

### Model Versioning vs Code Versioning
Code lives in Git. Models live in MLflow Registry. They're linked by tagging
the MLflow run with the Git SHA. This means for any prediction your API ever made,
you can find the exact training code, training data version, and hyperparameters.
That's reproducibility at a level that makes peer reviewers happy.


---
## ✅ Key Takeaways — CI/CD for Computational Toxicology

1. **Structure code for testability first**: separate functions that each do one thing — `smiles_to_mol`, `mol_to_fingerprint`, `featurize_smiles` — each can be tested in isolation.

2. **The quality gate is the most important CI step**: hardcoding `AUC ≥ 0.75` in the pipeline means a regression in data preprocessing or model code can never silently reach production.

3. **Tag everything with the git commit SHA**: Docker images, MLflow runs, and model files should all reference the exact commit that created them — this makes debugging and reproducing results trivial.

4. **Order your pipeline from cheapest to most expensive**: lint (2s) → unit tests (30s) → integration tests (2min) → build (5min) → deploy (10min). Fail fast before spending compute on later stages.

5. **Docker pins your scientific environment**: `requirements.txt` with pinned versions + a Dockerfile ensures the model runs identically on every machine, indefinitely.

6. **MLflow is your automated lab notebook**: every training run records parameters, metrics, and the model artifact. Future-you (and reviewers) will thank you.

---
### Complete File Structure
```
toxicity-qsar/
├── .github/workflows/
│   ├── ci.yml                 ← runs on every push
│   └── cd.yml                 ← runs on merge to main
├── src/
│   ├── featurize.py           ← SMILES → feature vectors
│   ├── model.py               ← training + evaluation
│   ├── train.py               ← entry point for CI training job
│   └── api.py                 ← FastAPI prediction service
├── tests/
│   ├── unit/
│   │   ├── test_featurize.py  ← unit tests (this notebook)
│   │   └── test_model.py
│   └── integration/
│       └── test_api.py        ← integration tests (this notebook)
├── Dockerfile                  ← multi-stage, non-root
├── requirements.txt            ← ALL versions pinned
└── mlflow_config.yml
```

---
*Part of the Python Ecosystem Tutorial Series | [hgoelgithub.github.io](https://hgoelgithub.github.io)*
